# Signal Protocol with LWE/SIS Hybridization

This notebook demonstrates the hybridization of Learning With Errors (LWE) and Short Integer Solution (SIS) problems for lightweight ephemeral key exchanges within the Signal Protocol. This mechanism aims to replace the classical X25519 ephemeral exchanges (DH3, DH4) in the X3DH handshake to provide post-quantum security.

In [1]:
import numpy as np
import time
import hashlib

## 1. LWE/SIS Hybridization Implementation

In [2]:
class LWESIS:
    def __init__(self, n=64, m=128, q=10007, alpha=0.005):
        self.n = n
        self.m = m
        self.q = q
        self.alpha = alpha
        # Shared public matrix M (can be derived from a public seed)
        np.random.seed(42) # For reproducibility in this demo
        self.M = np.random.randint(0, q, (n, m))
        
    def discrete_gaussian(self, size):
        return np.round(np.random.normal(0, self.alpha * self.q, size)).astype(int) % self.q
        
    def signal_function(self, y):
        """Hint algorithm S(y) to generate a signal."""
        return (y > (self.q // 4)) and (y < (3 * self.q // 4))
        
    def robust_extractor(self, x, sigma):
        """Extracts the shared bit."""
        return ((x + sigma * ((self.q - 1) // 2)) % self.q) % 2
        
    def generate_keypair_alice(self):
        """Alice's step: LWE"""
        s_A = np.random.randint(0, self.q, (1, self.n))
        e_A = self.discrete_gaussian((1, self.m))
        P_A = (s_A @ self.M + 2 * e_A) % self.q
        return s_A, P_A
        
    def generate_keypair_bob(self):
        """Bob's step: SIS"""
        s_B = self.discrete_gaussian((self.m, 1))
        P_B = (self.M @ s_B) % self.q
        return s_B, P_B
        
    def compute_shared_secret_alice(self, s_A, P_B):
        K_A = (s_A @ P_B) % self.q
        sigma = self.signal_function(K_A[0,0])
        root_key_alice = self.robust_extractor(K_A[0,0], sigma)
        return root_key_alice, sigma
        
    def compute_shared_secret_bob(self, s_B, P_A, sigma):
        K_B = (P_A @ s_B) % self.q
        root_key_bob = self.robust_extractor(K_B[0,0], sigma)
        return root_key_bob

## 2. Ephemeral Handshake Simulation

We simulate the replacement of the ephemeral DH exchange in X3DH with the LWE/SIS mechanism.

In [3]:
hybrid_kex = LWESIS()

# 1. Alice and Bob generate their ephemeral keys
s_A, P_A = hybrid_kex.generate_keypair_alice()
s_B, P_B = hybrid_kex.generate_keypair_bob()

# 2. Alice computes the shared secret and the reconciliation signal (sigma)
alice_shared_bit, sigma = hybrid_kex.compute_shared_secret_alice(s_A, P_B)

# 3. Alice sends P_A and sigma to Bob. Bob computes the shared secret.
bob_shared_bit = hybrid_kex.compute_shared_secret_bob(s_B, P_A, sigma)

print(f"Alice's Shared Bit: {alice_shared_bit}")
print(f"Bob's Shared Bit:   {bob_shared_bit}")
assert alice_shared_bit == bob_shared_bit, "Key agreement failed!"

Alice's Shared Bit: 0
Bob's Shared Bit:   0


## 3. Metric Analysis

In [4]:
def analyze_metrics():
    iterations = 100
    total_time_alice_gen = 0
    total_time_bob_gen = 0
    total_time_alice_compute = 0
    total_time_bob_compute = 0
    
    for _ in range(iterations):
        # Alice Generation
        start = time.time()
        s_A, P_A = hybrid_kex.generate_keypair_alice()
        total_time_alice_gen += (time.time() - start)
        
        # Bob Generation
        start = time.time()
        s_B, P_B = hybrid_kex.generate_keypair_bob()
        total_time_bob_gen += (time.time() - start)
        
        # Alice Compute
        start = time.time()
        alice_shared_bit, sigma = hybrid_kex.compute_shared_secret_alice(s_A, P_B)
        total_time_alice_compute += (time.time() - start)
        
        # Bob Compute
        start = time.time()
        bob_shared_bit = hybrid_kex.compute_shared_secret_bob(s_B, P_A, sigma)
        total_time_bob_compute += (time.time() - start)
        
    print(f"--- Average Execution Times ({iterations} iterations) ---")
    print(f"Alice Key Generation:  {total_time_alice_gen / iterations * 1000:.3f} ms")
    print(f"Bob Key Generation:    {total_time_bob_gen / iterations * 1000:.3f} ms")
    print(f"Alice Shared Compute:  {total_time_alice_compute / iterations * 1000:.3f} ms")
    print(f"Bob Shared Compute:    {total_time_bob_compute / iterations * 1000:.3f} ms")
    
    print("\n--- Key Sizes ---")
    # Size calculation based on numpy array shapes and data types (assuming int32 for simplicity in this demo)
    print(f"Alice Public Key (P_A): {P_A.nbytes} bytes")
    print(f"Bob Public Key (P_B):   {P_B.nbytes} bytes")
    print(f"Alice Secret Key (s_A): {s_A.nbytes} bytes")
    print(f"Bob Secret Key (s_B):   {s_B.nbytes} bytes")

analyze_metrics()

--- Average Execution Times (100 iterations) ---
Alice Key Generation:  0.050 ms
Bob Key Generation:    0.017 ms
Alice Shared Compute:  0.006 ms
Bob Shared Compute:    0.005 ms

--- Key Sizes ---
Alice Public Key (P_A): 1024 bytes
Bob Public Key (P_B):   512 bytes
Alice Secret Key (s_A): 512 bytes
Bob Secret Key (s_B):   1024 bytes


## 4. Security Analysis

### Post-Quantum Resilience
The hybridization of LWE (Learning With Errors) and SIS (Short Integer Solution) problems provides strong resistance against quantum attacks. Unlike ECDH, which relies on the Elliptic Curve Discrete Logarithm Problem (solvable by Shor's algorithm), LWE and SIS are believed to be hard even for quantum computers.

### Asymmetric Cost
This hybrid approach naturally fits the asymmetry of typical messaging environments. For instance, Bob (the responder) can perform the SIS step, which is highly efficient, while Alice (the initiator) performs the slightly heavier LWE step. This allows for lightweight ephemeral key exchanges that do not significantly burden battery-constrained mobile devices.